# 02 — Clean
Standardize the raw table inside DuckDB, run a quality report, and save interim Parquet.

Cleaning steps:
1. Normalize column names → `series`, `contestant`, `total_points`.
2. Cast `series`/`total_points` to INTEGER, TRIM the contestant name.
3. Validate structure: 21 series, exactly 5 contestants each, points in a sane range, no dupes.
4. Save `contestant_points_clean` to `data/interim/`.

In [ ]:
import sys, os
from pathlib import Path

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

import pandas as pd
from src.ingest import load_config
from src.clean_quality import (
    get_connection, run_sql, clean_table, quality_report,
    save_interim, load_to_duckdb,
)

cfg = load_config('config.yaml')
con = get_connection(cfg)
print(f'Project: {cfg["project_name"]}')

## Load raw table from DuckDB
(Re-loads from the CSV if the raw table isn't present, so this notebook can run standalone.)

In [ ]:
tables = {r[0] for r in con.execute('SHOW TABLES').fetchall()}
if 'contestant_points_raw' not in tables:
    from src.ingest import load_raw_csv
    load_to_duckdb(load_raw_csv(cfg, 'taskmaster_data.csv'), 'contestant_points_raw', con)

df_raw = con.execute('SELECT * FROM contestant_points_raw').df()
df_raw.head()

## Rename + cast + trim (in DuckDB)
`clean_table` runs the cast/trim as SQL against a staging table and returns the result.

In [ ]:
# Normalize the three columns to snake_case names via a SQL projection.
df_renamed = run_sql(
    '''
    SELECT
        "Series Number"   AS series,
        "Contestant Name" AS contestant,
        "Total Points"    AS total_points
    FROM contestant_points_raw
    ''',
    con,
)

# Cast types, trim the name, drop exact-duplicate rows.
df_clean = clean_table(
    df_renamed,
    table_name='contestant_points_stage',
    con=con,
    cast_map={'series': 'INTEGER', 'total_points': 'INTEGER'},
    strip_columns=['contestant'],
)
df_clean = df_clean.sort_values(['series', 'total_points'], ascending=[True, False]).reset_index(drop=True)
df_clean.head()

## Validation checks
Assert the structure we expect before saving.

In [ ]:
# 1. Type / null checks via the standard quality report.
qc = quality_report(df_clean, 'contestant_points_clean', con,
                    required_columns=['series', 'contestant', 'total_points'],
                    max_null_pct=0.0)

# 2. Structural assertions specific to this dataset.
assert len(df_clean) == 105, f'expected 105 rows, got {len(df_clean)}'
assert df_clean['series'].nunique() == 21, 'expected 21 distinct series'

per_series = df_clean.groupby('series').size()
assert (per_series == 5).all(), f'every series must have 5 contestants; got:\n{per_series[per_series != 5]}'

# 3. No duplicate contestant within a series.
dup_in_series = df_clean.duplicated(subset=['series', 'contestant']).sum()
assert dup_in_series == 0, f'{dup_in_series} duplicate contestant(s) within a series'

# 4. Points look sane (positive, within observed Taskmaster range).
assert df_clean['total_points'].between(1, 300).all(), 'total_points out of expected range'

print('\n✓ All structural checks passed: 21 series x 5 contestants, no dupes, points in range.')

In [ ]:
# Quick look at the spread of raw totals by series length — shows WHY we must normalize.
run_sql(
    '''
    SELECT series,
           COUNT(*)          AS contestants,
           SUM(total_points) AS series_total,
           MIN(total_points) AS min_pts,
           MAX(total_points) AS max_pts
    FROM contestant_points_stage
    GROUP BY series
    ORDER BY series
    ''',
    con,
)

## Persist clean table + save interim Parquet

In [ ]:
load_to_duckdb(df_clean, 'contestant_points_clean', con)
save_interim(df_clean, cfg, 'contestant_points_clean.parquet')

---
**Next:** `03-prepare.ipynb` — add share-of-series-points and apples-to-apples metrics, then export.

---
## Cleanup

In [ ]:
con.close()
print('connection closed')